# AI for Market Trend Analysis

This notebook implements an AI-based approach to analyze and predict market trends using deep learning techniques, specifically LSTM (Long Short-Term Memory) networks.

## 1. Setting up Time Series Data

First, we'll import the required libraries and fetch historical market data using yfinance.

### Stock Selection
You can analyze any publicly traded stock by entering its symbol. Here are some examples:
- AAPL (Apple Inc.)
- MSFT (Microsoft Corporation)
- GOOGL (Alphabet Inc.)
- AMZN (Amazon.com Inc.)
- TSLA (Tesla Inc.)
- META (Meta Platforms Inc.)
- NFLX (Netflix Inc.)

The program will fetch 1000 days of historical data for the selected stock.

In [7]:
# Import required libraries
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.style.use('fivethirtyeight')

In [8]:
# Get user input for stock symbol
symbol = input("Enter the stock symbol (e.g., AAPL for Apple, MSFT for Microsoft, GOOGL for Google): ").upper()
start_date = datetime.now() - timedelta(days=1000)
end_date = datetime.now()

try:
    # Fetch historical data
    df = yf.download(symbol, start=start_date, end=end_date)
    if len(df) == 0:
        raise Exception("No data found for the symbol")
    print("Downloaded {} rows of data for {}".format(len(df), symbol))
    print("\nFirst few rows of the data:")
    display(df.head())
    
    # Display basic statistics
    print("\nBasic statistics:")
    print("Date Range: {} to {}".format(df.index.min(), df.index.max()))
    print("Current Stock Price: ${:.2f}".format(df['Close'][-1]))
    print("Highest Price: ${:.2f}".format(df['High'].max()))
    print("Lowest Price: ${:.2f}".format(df['Low'].min()))
except Exception as e:
    print("Error: Could not fetch data for {}. Please check if the symbol is correct.".format(symbol))

C:\Users\Admin\AppData\Local\Temp\ipykernel_35328\2343106704.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Downloaded 684 rows of data for AAPL

First few rows of the data:


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2022-12-08,140.666153,141.524064,139.137721,140.380193,62128300
2022-12-09,140.182983,143.545564,138.940497,140.360473,76097000
2022-12-12,142.480576,142.490431,139.098269,140.715461,70462700
2022-12-13,143.446960,147.884379,142.234070,147.420914,93886200
2022-12-14,141.218384,144.620402,139.196890,143.328622,82291200



Basic statistics:
Date Range: 2022-12-08 00:00:00 to 2025-09-02 00:00:00
Error: Could not fetch data for AAPL. Please check if the symbol is correct.


## 2. Data Preprocessing and Feature Engineering

Now we'll prepare our data by calculating technical indicators and scaling the features.

In [9]:
def calculate_technical_indicators(df):
    # Calculate Moving Averages
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['MA50'] = df['Close'].rolling(window=50).mean()
    
    # Calculate RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Calculate MACD
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    return df

# Apply technical indicators
df = calculate_technical_indicators(df)

# Drop any NaN values
df = df.dropna()
df.head()

Price,Close,High,Low,Open,Volume,MA20,MA50,RSI,MACD,Signal_Line
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,,,,,
Date,,,,,,,,,,
2023-02-21,146.638611,149.423646,146.569487,148.337281,58867200,147.482571,138.330520,56.786621,3.715355,4.123338
2023-02-22,147.063309,148.090405,145.335011,147.023796,51011300,147.808345,138.458463,55.823667,3.278464,3.954363
2023-02-23,147.547211,148.475556,145.414009,148.228656,48394200,148.191348,138.605748,47.787066,2.937410,3.750972
2023-02-24,144.890579,145.364622,143.912851,145.285613,55469600,148.337980,138.653948,35.419168,2.424803,3.485738
2023-02-27,146.085571,147.320069,145.621399,145.878184,44998500,148.447231,138.706720,42.651441,2.090882,3.206767


In [10]:
# Scale the features
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define features to scale
features = ['Close', 'Volume', 'MA20', 'MA50', 'RSI', 'MACD', 'Signal_Line']

# Verify all features exist in the dataframe
missing_features = [f for f in features if f not in df.columns]
if missing_features:
    raise ValueError(f"Missing features in dataframe: {missing_features}")

# Check for any infinite or NaN values
if df[features].isin([np.inf, -np.inf]).any().any():
    print("Warning: Infinite values found in features. Replacing with NaN...")
    df[features] = df[features].replace([np.inf, -np.inf], np.nan)

if df[features].isna().any().any():
    print("Warning: NaN values found. Filling with forward fill method...")
    df[features] = df[features].fillna(method='ffill')
    # If any remaining NaN at the beginning, fill with backward fill
    df[features] = df[features].fillna(method='bfill')

# Perform scaling
try:
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df[features]), 
        columns=features, 
        index=df.index
    )
    print("Features scaled successfully!")
    print("\nScaled data summary:")
    display(df_scaled.describe())
except Exception as e:
    raise Exception(f"Error during scaling: {str(e)}")

Features scaled successfully!

Scaled data summary:


,Close,Volume,MA20,MA50,RSI,MACD,Signal_Line
count,6.350000e+02,6.350000e+02,6.350000e+02,6.350000e+02,6.350000e+02,6.350000e+02,635.000000
mean,4.475860e-16,-2.797412e-17,1.790344e-16,5.371032e-16,4.923446e-16,-1.118965e-17,0.000000
std,1.000788e+00,1.000788e+00,1.000788e+00,1.000788e+00,1.000788e+00,1.000788e+00,1.000788
min,-2.108944e+00,-1.367788e+00,-1.932563e+00,-2.161773e+00,-2.624011e+00,-3.556069e+00,-3.061296
25%,-8.282899e-01,-5.333053e-01,-8.179269e-01,-7.235479e-01,-7.411555e-01,-7.101008e-01,-0.692251
50%,-2.003850e-01,-2.282281e-01,-2.264270e-01,-2.706444e-01,9.523696e-02,9.012419e-02,0.088132
75%,9.081351e-01,2.216272e-01,9.497177e-01,9.498778e-01,7.602628e-01,6.614316e-01,0.691712
max,2.303485e+00,1.060970e+01,2.026586e+00,1.698611e+00,2.382448e+00,2.491205e+00,2.471002


## 3. Building LSTM Model

Let's implement our LSTM model using TensorFlow/Keras.

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length, 0])  # Predicting the Close price
    return np.array(X), np.array(y)

# Create sequences
seq_length = 60  # 60 days of historical data
X, y = create_sequences(df_scaled.values, seq_length)

# Build LSTM model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(seq_length, len(features))),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 50)         │        11,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,851 (124.42 KB)

 Trainable params: 31,851 (124.42 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Training and Validation

Now we'll split our data and train the model.

In [12]:
# Split data into training and validation sets
train_split = 0.8
split_idx = int(len(X) * train_split)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

# Implement early stopping and model checkpointing
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 96ms/step - loss: 0.3327 - val_loss: 0.2315
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 96ms/step - loss: 0.3327 - val_loss: 0.2315
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - loss: 0.0940 - val_loss: 0.1453
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - loss: 0.0940 - val_loss: 0.1453
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0711 - val_loss: 0.1873
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0711 - val_loss: 0.1873
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - loss: 0.0650 - val_loss: 0.1594
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - loss: 0.0650 - val_loss: 0.1594
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 0.0566 - val_loss: 0.1809
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 0.0566 - val_loss: 0.1809
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 0.0534 - val_loss: 0.1612
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - 

## 5. Making Market Predictions

Let's use our trained model to make predictions.

In [13]:
# Make predictions
train_predictions = model.predict(X_train)
val_predictions = model.predict(X_val)

# Inverse transform predictions
def inverse_transform_predictions(predictions):
    # Create a dummy array with the same shape as the original features
    dummy = np.zeros((len(predictions), len(features)))
    dummy[:, 0] = predictions.flatten()  # Put predictions in the first column (Close price)
    return scaler.inverse_transform(dummy)[:, 0]

train_predictions = inverse_transform_predictions(train_predictions)
val_predictions = inverse_transform_predictions(val_predictions)
actual_values = inverse_transform_predictions(y.reshape(-1, 1))

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


## 6. Visualization and Analysis

Finally, let's visualize our results and calculate performance metrics.

In [14]:
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

# Calculate performance metrics
val_rmse = math.sqrt(mean_squared_error(actual_values[split_idx:], val_predictions))
val_mae = mean_absolute_error(actual_values[split_idx:], val_predictions)

print('Validation RMSE: ${:.2f}'.format(val_rmse))
print('Validation MAE: ${:.2f}'.format(val_mae))

# Create interactive plot
fig = go.Figure()

# Add actual values
fig.add_trace(go.Scatter(
    x=df.index[seq_length:],
    y=actual_values,
    mode='lines',
    name='Actual',
    line=dict(color='blue')
))

# Add training predictions
fig.add_trace(go.Scatter(
    x=df.index[seq_length:split_idx+seq_length],
    y=train_predictions,
    mode='lines',
    name='Training Predictions',
    line=dict(color='green')
))

# Add validation predictions
fig.add_trace(go.Scatter(
    x=df.index[split_idx+seq_length:],
    y=val_predictions,
    mode='lines',
    name='Validation Predictions',
    line=dict(color='red')
))

fig.update_layout(
    title='{} Stock Price Prediction'.format(symbol),
    xaxis_title='Date',
    yaxis_title='Price',
    template='plotly_white'
)

fig.show()

Validation RMSE: $9.90
Validation MAE: $7.01
